# 03 — Customer churn analysis in a notebook

This notebook walks through a typical supervised ML workflow using OpenDataSci as the agent:
profile a dataset, build a churn classifier, and interpret the predictions.

Jupyter supports **top-level `await`**, so you can call `await agent.astream(...)` directly
in cells — no `asyncio.run()` wrapper needed. If you are running this outside Jupyter,
see `02_script.py` for the script pattern.

**Prerequisites:** `pip install open-data-sci` and `ANTHROPIC_API_KEY` set in your environment.

## Create a sample dataset

The cell below generates a synthetic customer churn dataset so this notebook is
fully self-contained. Skip it and update the path below if you have your own data.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n   = 2_000

age            = rng.integers(20, 65, n)
tenure         = rng.integers(1, 120, n)      # months as a customer
monthly_charge = rng.uniform(20, 150, n).round(2)
support_calls  = rng.integers(0, 10, n)       # support contacts in the past year
has_contract   = rng.choice([0, 1], n, p=[0.4, 0.6])

log_odds = (
    -1.5
    + 0.25  * support_calls
    - 0.012 * tenure
    + 0.008 * monthly_charge
    - 0.9   * has_contract
)
prob    = 1 / (1 + np.exp(-log_odds))
churned = rng.binomial(1, prob).astype(bool)

df = pd.DataFrame({
    "age":            age,
    "tenure_months":  tenure,
    "monthly_charge": monthly_charge,
    "support_calls":  support_calls,
    "has_contract":   has_contract,
    "churned":        churned,
})
df.to_csv("churn.csv", index=False)
print(f"churn.csv written — {len(df):,} rows, {df['churned'].mean():.1%} churn rate")

## Load data into OpenDataSci

`create_agent` is the main entry point for scripted use. Pass a file path and an
optional `OpenDataSciConfig`; it returns an `Agent` that boots its sandbox
when you enter it as an async context manager.

To keep the agent open across notebook cells we enter it via an `AsyncExitStack`
(create → work → close) rather than an `async with` block, which would require all
queries to live in a single cell.

In [ ]:
from contextlib import AsyncExitStack

from opendatasci import OpenDataSciConfig, create_agent

cfg = OpenDataSciConfig(
    provider="anthropic",         # or "openai", "gemini", "ollama", …
    # model="claude-sonnet-4-6",  # omit to use the provider default
    temperature=0.1,              # low temperature → more deterministic code and analysis
)

# create_agent() is synchronous; entering the agent boots its sandbox. An
# AsyncExitStack keeps the agent open across cells and closes it in the cleanup cell.
stack = AsyncExitStack()
agent = await stack.enter_async_context(create_agent("churn.csv", config=cfg))
print(f"Agent ready — provider: {cfg.provider}, model: {cfg.model}")

## Streaming helper

`astream()` yields `AgentStreamEvent` objects as the agent works.
`token` events are individual text chunks; `response` carries the final assembled reply.

The helper below prints tokens as they arrive (streaming output) and
returns the complete response string so you can inspect or save it.

In [ ]:
async def ask(query: str) -> str:
    """Stream a query and return the final response text."""
    response = ""
    async for event in agent.astream(query):
        if event.type == "token":
            print(event.content, end="", flush=True)
        elif event.type == "response":
            response = event.content
            print()  # trailing newline after the streamed tokens
        elif event.type == "error":
            print(f"\n[error] {event.content}")
            break
    return response

## Turn 1 — profile the data

Before touching any models, ask for a structured profile of the dataset.
OpenDataSci will check column types, null counts, value distributions, and class balance.

In [ ]:
eda = await ask(
    "Profile this dataset. I want:"
    " (1) schema with dtype and null count per column,"
    " (2) class balance for the target (churned),"
    " (3) summary statistics for each numeric feature,"
    " (4) any data quality flags you notice."
)

## Turn 2 — build a classifier

Ask for a proper ML pipeline: OpenDataSci will choose model families, split the data,
tune hyperparameters with cross-validation, and report evaluation metrics.

In [ ]:
model_report = await ask(
    "Build a churn classifier. Requirements:"
    " stratified 80/20 train-test split,"
    " try at least two model families,"
    " optimise for ROC-AUC,"
    " report precision / recall / F1 at the default threshold plus the AUC."
    " Recommend which model to use in production and explain why."
)

## Turn 3 — interpret the predictions

OpenDataSci keeps the full conversation in context, so follow-up questions don't need
to repeat any setup. Here we ask for SHAP-based feature importance on the best model.

In [ ]:
interpretation = await ask(
    "For the best model:"
    " (1) compute SHAP values and plot mean absolute SHAP per feature,"
    " (2) name the top 3 drivers of churn and explain each in one sentence"
    " a non-technical stakeholder could understand."
)

## Compact a long session

After many turns the context window fills up. `compact_chat_history()` asks the LLM
to summarise the history and replaces the full transcript with the summary,
freeing space for more analysis without losing continuity.

In [ ]:
# Uncomment when your session gets long:
# summary = await agent.compact_chat_history()
# print(summary)

## Cleanup

Release sandbox resources when you are done.

In [ ]:
await stack.aclose()  # exits the agent context and tears down its sandbox